# Advanced: Generative Modern Hopfield Sampling and Why It Reproduces Stylized Market Facts

A central question in synthetic market generation is straightforward: how can a memory-based sampler reproduce realistic return structure without hand-coding each stylized fact one-by-one?

This notebook presents the derivation-first organization of the `L6d` method. For executable implementation details and the full lab workflow, see [L6d Lab Solution](CHEME-5820-L6d-Lab-Solution-Spring-2026.ipynb).


> __Learning Objectives:__
>
> By the end of this notebook, you should be able to:
>
> * __Derive the stateful generator map:__ Track each state variable in the synthetic-day loop and show how one new return vector is produced from the current query state.
> * __Explain mechanism-to-metric links:__ Identify which components of the sampler primarily control tail behavior, linear return autocorrelation, and volatility clustering.
> * __Tune with explicit trade-offs:__ Use parameter-role logic to decide when to favor regime persistence, sampling diversity, or marginal-moment alignment.


Let's get started!

___


## Method Construction

Before writing equations, fix the design objective: generate synthetic returns that remain memory-faithful, avoid deterministic collapse, and exhibit realistic volatility dynamics without forcing strong linear autocorrelation in return levels.

> __Intuition first:__
>
> The sampler is intentionally not memoryless. A memoryless map can match one-day histograms but often fails to reproduce clustered volatility episodes.
>
> * __Why state is needed:__ The query state carries information from previous synthetic days, so the selected memory mixture can evolve gradually instead of jumping independently each day.
> * __Why regime persistence appears:__ Sticky mixture weights and a persistent latent volatility state induce multi-day dependence in return magnitudes, even when linear return autocorrelation remains small.
>
> The derivation below makes these design choices explicit.


Before constructing the update equations, define notation so each symbol has one meaning.

> __Notation guide (used throughout):__
>
> Let $F$ be the number of firms (cross-sectional features) and $K$ be the number of historical memory days. The memory matrix is $\boldsymbol{M}\in\mathbb{R}^{F\times K}$, where column $j$ is the return vector for memory day $j$. The query state at synthetic time $t$ is $\boldsymbol{s}_t\in\mathbb{R}^{F}$, and the recovered memory probability vector is $\boldsymbol{p}_t\in\Delta^{K-1}$.
>
> We use $\boldsymbol{q}^{\mathrm{new}}_t$ for the top-$k$ proposal, $\boldsymbol{q}_t$ for sticky/sharpened sampling weights, and $\boldsymbol{m}_t$ for the generated synthetic return vector. A scalar latent volatility state $h_t>0$ modulates amplitude through $\sqrt{h_t}$.

With notation fixed, we can now build the sampler as a sequence of justified modeling moves.

> __Generative Sampler Construction (L6d state update form):__
>
> We begin with retrieval. Given the current query state $\boldsymbol{s}_t$, recover a probability vector over historical memory days:
> $$
> \boldsymbol{p}_{t}=\mathrm{RecoverProbabilities}(\boldsymbol{s}_{t};\beta,\epsilon_{p},\epsilon_{s},\mathrm{maxiterations}).
> $$
> A direct deterministic map $\boldsymbol{M}\boldsymbol{p}_t$ is memory-faithful, but by itself it can become brittle: trajectories may collapse onto a narrow set of memories and react too sharply to small query perturbations.
>
> To preserve high-probability memories while maintaining diversity, we perturb the recovered probabilities and keep only a sparse top-$k$ support:
> $$
> \tilde{p}_{t,j}=p_{t,j}+\sigma_{\mathrm{select}}\,\varepsilon_{t,j},\qquad \varepsilon_{t,j}\sim\mathcal{N}(0,1),
> $$
> $$
> q^{\mathrm{new}}_{t,j}=\begin{cases}
> p_{t,j}, & j\in\mathcal{I}_{t},\\
> 0, & j\notin\mathcal{I}_{t},
> \end{cases}
> \qquad
> \boldsymbol{q}^{\mathrm{new}}_{t}\leftarrow\frac{\boldsymbol{q}^{\mathrm{new}}_{t}}{\sum_{j=1}^{K}q^{\mathrm{new}}_{t,j}+10^{-12}},
> $$
> where $\mathcal{I}_{t}$ is the index set of the top-$k$ values of $\tilde{p}_{t,j}$. This construction introduces controlled exploration without discarding retrieval structure.
>
> Sparse selection alone is still noisy across time, so we add temporal persistence in the mixture weights:
> $$
> \boldsymbol{q}_{t}=\lambda_{q}\,\boldsymbol{q}_{t-1}+(1-\lambda_{q})\,\boldsymbol{q}^{\mathrm{new}}_{t},
> \qquad
> \boldsymbol{q}_{t}\leftarrow\frac{\boldsymbol{q}_{t}}{\sum_{j=1}^{K}q_{t,j}+10^{-12}}.
> $$
> This sticky update behaves like a low-pass filter in mixture space, preventing implausible day-to-day regime switching.
>
> We then control regime concentration with sharpening:
> $$
> q_{t,j}\leftarrow q_{t,j}^{\gamma_{\mathrm{sharpen}}},
> \qquad
> \boldsymbol{q}_{t}\leftarrow\frac{\boldsymbol{q}_{t}}{\sum_{j=1}^{K}q_{t,j}+10^{-12}}.
> $$
> For $\gamma_{\mathrm{sharpen}}>1$, mass is concentrated on dominant memories; for $\gamma_{\mathrm{sharpen}}\approx 1$, mixtures remain smoother.
>
> Given these persistent weights, the raw synthetic day is:
> $$
> \boldsymbol{m}^{\mathrm{raw}}_{t}=\boldsymbol{M}\boldsymbol{q}_{t}.
> $$
> This preserves cross-sectional memory structure, but it does not by itself create persistent volatility episodes.
>
> To introduce magnitude clustering without forcing strong linear dependence in return levels, we evolve a latent log-volatility state:
> $$
> \log h_{t}=\rho_{\log h}\,\log h_{t-1}+\sigma_{\log h}\,\xi_{t},\qquad \xi_{t}\sim\mathcal{N}(0,1),
> $$
> $$
> h_{t}=\exp(\log h_{t}),
> \qquad
> \boldsymbol{m}_{t}=\boldsymbol{\mu}_{\mathrm{orig}}+\sqrt{h_{t}}\left(\boldsymbol{m}^{\mathrm{raw}}_{t}-\boldsymbol{\mu}_{\mathrm{orig}}\right).
> $$
> Centering around $\boldsymbol{\mu}_{\mathrm{orig}}$ keeps unconditional location stable, while $h_t$ controls time-varying amplitude.
>
> Finally, we close the state loop by updating the next query state:
> $$
> \boldsymbol{s}_{t+1}=\lambda_{\mathrm{query}}\,\boldsymbol{s}_{t}+(1-\lambda_{\mathrm{query}})\,\boldsymbol{m}_{t}+\sigma_{\mathrm{query}}\,\boldsymbol{\eta}_{t},
> \qquad
> \boldsymbol{\eta}_{t}\sim\mathcal{N}(\boldsymbol{0},\boldsymbol{I}_{F}).
> $$
> This separates two dependence channels: $\lambda_{\mathrm{query}}$ governs persistence in return levels, while $\rho_{\log h}$ governs persistence in return magnitudes.
>
> Mean matching and partial standard-deviation matching may be applied after synthesis as post-calibration layers. These operations align marginals but do not independently create clustered volatility.

The resulting map is intentionally modular: retrieval defines candidate regimes, mixture dynamics define regime persistence, and latent volatility defines magnitude clustering.


## Stylized-Facts Interpretation

Now connect model mechanisms to empirical diagnostics.

> __Diagnostic definitions:__
>
> For a scalar return series $r_t$ (or one projected portfolio return), define lag-$\ell$ dependence metrics
> $$
> \rho_{r}(\ell)=\mathrm{Corr}(r_t,r_{t-\ell}),\qquad
> \rho_{|r|}(\ell)=\mathrm{Corr}(|r_t|,|r_{t-\ell}|),\qquad
> \rho_{r^2}(\ell)=\mathrm{Corr}(r_t^2,r_{t-\ell}^2).
> $$
> Stylized-fact matching usually targets small $\rho_r(1)$ with positive short-lag $\rho_{|r|}(\ell)$ and $\rho_{r^2}(\ell)$.

With these diagnostics fixed, the L6d mechanism map is:

> __Mechanism-to-outcome map:__
>
> * __Fat tails:__ Stochastic top-$k$ selection, sharpening, and volatility scaling produce regime switching and occasional amplitude bursts. This combination generates heavier tails than a single Gaussian linear map.
> * __Low linear autocorrelation in returns:__ Keeping $\lambda_{\mathrm{query}}$ and $\lambda_q$ moderate (or small) limits direct carry-over in return levels, so $\rho_r(1)$ can remain near zero.
> * __Volatility clustering:__ Persistent log-volatility dynamics,
> $$
> \log h_t=\rho_{\log h}\log h_{t-1}+\sigma_{\log h}\xi_t,
> $$
> imply that large $h_t$ values tend to be followed by large $h_{t+1}$ values, creating positive short-lag dependence in $|r_t|$ and $r_t^2$.

A practical interpretation is that return-level dependence and magnitude dependence come from different parameter groups. This is why tuning by a single aggregate score often obscures which mechanism is failing.


## Practical Tuning Guide

Treat calibration as a staged identification problem: fit dynamic structure first, then apply moment corrections conservatively.

> __Parameter-role map:__
>
> * __Volatility persistence (`rho_logh`) and shock scale (`sigma_logh`):__ Increase `rho_logh` to lengthen volatility episodes; increase `sigma_logh` to broaden episode intensity.
> * __Mixture memory (`lambda_q`) and query persistence (`lambda_query`):__ Larger values increase temporal carry-over. Keep these controlled when you want weak linear return autocorrelation.
> * __Selection granularity (`k`) and concentration (`gamma_sharpen`):__ Larger `k` smooths regimes; larger `gamma_sharpen` concentrates mass and increases regime contrast.
> * __Selection noise (`sigma_select`):__ Small positive values prevent deterministic collapse; overly large values destabilize regime continuity.
> * __Post-calibration controls (`match_mean`, `match_std`, `alpha_std`):__ Use only after dynamics are acceptable; these are alignment controls, not dynamic generators.

A stable workflow is:

1. Tune `rho_logh` and `sigma_logh` until magnitude autocorrelation diagnostics are in the desired range.
2. Adjust `lambda_q`, `lambda_query`, `k`, and `gamma_sharpen` to balance regime smoothness versus diversity while monitoring $\rho_r(1)$.
3. Apply mean and variance corrections lightly to align marginals without destroying distribution shape.

This sequence prevents overfitting marginals before dynamic structure is correct.


## Summary

This notebook organizes the L6d generator as a derivation-first, state-space map so each stylized fact can be traced to a specific mechanism block.


> __Key Takeaways:__
>
> * __Stateful dynamics drive clustering:__ Heavy tails can appear in static mixtures, but persistent volatility states are the direct mechanism behind clustered magnitudes.
> * __Dependence metrics should be split:__ Evaluate return autocorrelation and magnitude autocorrelation separately because they are controlled by different parameters.
> * __Calibration is two-stage:__ First fit dynamic behavior (regime persistence and volatility structure), then apply marginal-moment corrections as a final alignment layer.


___


### References
1. Hopfield JJ (1982), *Neural networks and physical systems with emergent collective computational abilities*, PNAS 79(8):2554-2558. DOI: https://doi.org/10.1073/pnas.79.8.2554
   - Supports the energy-based attractor perspective underlying memory retrieval interpretations.

2. Ramsauer H, Schafl B, Lehner J, et al. (2021), *Hopfield Networks is All You Need*, ICLR 2021. arXiv: https://arxiv.org/abs/2008.02217
   - Supports modern Hopfield retrieval as probabilistic memory selection used in the L6d construction.

3. Cont R (2001), *Empirical properties of asset returns: stylized facts and statistical issues*, Quantitative Finance 1(2):223-236. DOI: https://doi.org/10.1080/713665670
   - Supports diagnostic targets for fat tails, weak linear autocorrelation in returns, and volatility clustering.

Note: the sampler equations above are method-specific to the CHEME-5820 L6d workflow and are documented for reproducible tuning logic.
